# Bagging-based feature selection

This notebook finds a feature set independently for every power-quality disturbance (PQD) class. It follows the idea in `Vossi_01_Task.ipynb`: start with all eight features and remove features that do not improve classification, while evaluating the decision across bootstrap samples and multiple classifiers.

The word *bagging* refers to bootstrap aggregation of the evaluation. Each candidate feature set is trained on a sample drawn with replacement and scored on its out-of-bag (OOB) samples. Each class is evaluated as a one-vs-rest binary problem, so the output preserves class-specific feature choices.


## 1. Imports and paths

The helper module contains reusable data-loading and feature-selection functions. The path setup makes the notebook work both from `Code/01 Task` and from the repository root.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

TASK_DIR = Path.cwd()
if TASK_DIR.name != "01 Task":
    TASK_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "01 Task")
sys.path.insert(0, str(TASK_DIR))

from bagging_feature_selection import load_dataset, run_feature_selection


## 2. Load and inspect the data

Each CSV file represents one disturbance class; its filename supplies the class label. The loader standardizes the columns to `F1` through `F8`, adds the source noise level, and combines all requested SNR folders. The noise-level column is retained for diagnostics but is not passed to the classifiers.


In [ ]:
folders = [TASK_DIR / name for name in (
    "SNR_20db", "SNR_30db", "SNR_40db", "SNR_50db", "SNR_noiseless"
)]
data = load_dataset(folders)
print(f"Loaded {len(data):,} samples from {data['noise_level'].nunique()} noise levels")
data.head()


## 3. Select features for every class and classifier

For each class and classifier pair, the selector creates a one-vs-rest target: the selected class is positive and all other classes are negative. It starts with all eight features, evaluates every possible one-feature removal using bootstrap OOB F1, accepts the best removal when it is not meaningfully worse than the current set, and repeats until no further removal is supported or one feature remains.

The scaler is inside the estimator pipeline, so it is fitted separately for every bootstrap training sample. This prevents OOB information from leaking into preprocessing. `min_improvement=0.002` prevents tiny score fluctuations from causing unstable removals. Increase `n_bootstraps` for a final experiment.


In [ ]:
results = run_feature_selection(
    data,
    n_bootstraps=20,
    min_improvement=0.002,
    random_state=42,
)

summary = pd.DataFrame({
    "target_label": [result.target_label for result in results],
    "classifier": [result.classifier for result in results],
    "selected_features": [list(result.selected_features) for result in results],
    "bootstrap_f1": [result.score for result in results],
})
summary


## 4. Inspect the per-class recommendations

There is no single overall feature set in this workflow. The table contains one row for every class and classifier, so selected features can differ between disturbance types. This preserves class-specific information that would be hidden by aggregating all classes into one consensus.


In [ ]:
for target_label in summary["target_label"].unique():
    print(f"\n{target_label}")
    display(summary[summary["target_label"] == target_label])


## 5. Interpret the output

The summary reports the selected subset and mean bootstrap one-vs-rest F1 for each classifier and target class. A high score means that the selected class can be separated from the other disturbance types using that subset.

Use the rows for the relevant target class when building a class-specific downstream model. For a final reported result, rerun with a larger `n_bootstraps` value and keep the random seed fixed so the experiment remains reproducible.
